In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta, timezone

import os
import sys
sys.path.append(os.path.abspath('./src'))

from db_functions import DotaDB

db = DotaDB()

In [8]:
team_history = pd.concat([rad, dire]).sort_values(['teamId', 'startDateTime'])
team_history

,id,teamId,won,startDateTime
78939,8115561069,-2147475696,True,2025-01-05 21:01:41
117182,6819143135,-2147135186,True,2022-10-23 12:08:18
117191,6819319015,-2147135186,False,2022-10-23 13:44:36
109306,8639785158,-2146807748,False,2026-01-07 21:01:24
109254,8639303523,-2146620491,False,2026-01-07 14:45:11
...,...,...,...,...
112982,8700141994,10061897,False,2026-02-21 14:09:13
113076,8701690529,10067366,True,2026-02-22 13:34:20
113067,8701677391,10067419,True,2026-02-22 13:26:39
113084,8701819849,10067419,True,2026-02-22 14:48:33


In [7]:
## Calculate form: for now, wins out of last 5 games, simple int
query = 'SELECT * FROM match_details'
df = pd.DataFrame(db.query_select_to_df(query, table='match_details'))
df = df.sort_values(by='startDateTime', axis=0, ascending=False)
rad = df[['id', 'radiantTeamId', 'didRadiantWin', 'startDateTimeHuman']].rename(
    columns={'radiantTeamId': 'teamId', 'didRadiantWin': 'won', 'startDateTimeHuman': 'startDateTime'}
)
dire = df[['id', 'direTeamId', 'didRadiantWin', 'startDateTimeHuman']].rename(
    columns={'direTeamId': 'teamId', 'didRadiantWin': 'won', 'startDateTimeHuman': 'startDateTime'}
)
dire['won'] = ~dire['won'] 

team_history = pd.concat([rad, dire]).sort_values(['teamId', 'startDateTime'])

team_history['form'] = (
    team_history.groupby('teamId')['won']
    .apply(lambda x: x.shift(1).rolling(window=5, min_periods=1).sum())
    .reset_index(level=0, drop=True)
)

df = df.merge(
    team_history[['id', 'teamId', 'form']], 
    left_on=['id', 'radiantTeamId'], 
    right_on=['id', 'teamId'], 
    how='left'
).rename(columns={'form': 'radiantForm'}).drop('teamId', axis=1)

df = df.merge(
    team_history[['id', 'teamId', 'form']], 
    left_on=['id', 'direTeamId'], 
    right_on=['id', 'teamId'], 
    how='left'
).rename(columns={'form': 'direForm'}).drop('teamId', axis=1)

In [10]:
## Roster longevity
##TODO: hero longevity and othe ideas
query = '''
    SELECT match_id, mp."heroId", mp.position, mp."isRadiant", mp."isVictory",
        variant, mp."steamAccountId", name, mp."realName",
        md."radiantTeamId", md."direTeamId", md."startDateTime"
    FROM match_players mp
    INNER JOIN match_details md
    ON md.id = mp.match_id
'''
results = db.query_select(query) 
df_roster = pd.DataFrame(
    results, 
    columns=[
        'matchId', 
        'heroId', 
        'position',
        'isRadiant', 
        'isVictory', 
        'variant', 
        'steamAccountId', 
        'name', 
        'realName', 
        'radiantTeamId', 
        'direTeamId', 
        'startDateTime'
    ]
)
df_roster['realName'].value_counts() ## Most of them don't have any
df_roster = df_roster.drop('realName', axis=1).sort_values(by='startDateTime', ascending=False)
roster_groups = (
    df_roster.sort_values(['matchId', 'isRadiant', 'steamAccountId'])
    .groupby(['matchId', 'isRadiant'])['steamAccountId']
    .apply(tuple)
    .reset_index()
)

roster_groups.columns = ['matchId', 'isRadiant', 'roster_tuple']

df = df.merge(roster_groups[roster_groups['isRadiant'] == True], left_on='id', right_on='matchId', how='left')
df = df.rename(columns={'roster_tuple': 'rad_roster_tuple'}).drop('isRadiant', axis=1)

df = df.merge(roster_groups[roster_groups['isRadiant'] == False], left_on='id', right_on='matchId', how='left')
df = df.rename(columns={'roster_tuple': 'dire_roster_tuple'}).drop('isRadiant', axis=1)

rad_rosters = df[['id', 'radiantTeamId', 'rad_roster_tuple', 'startDateTime']].rename(
    columns={'radiantTeamId': 'teamId', 'rad_roster_tuple': 'roster'}
)
dire_rosters = df[['id', 'direTeamId', 'dire_roster_tuple', 'startDateTime']].rename(
    columns={'direTeamId': 'teamId', 'dire_roster_tuple': 'roster'}
)

roster_history = pd.concat([rad_rosters, dire_rosters]).sort_values(['teamId', 'roster', 'startDateTime'])
roster_history['roster_experience'] = roster_history.groupby(['teamId', 'roster']).cumcount()

df = df.merge(
    roster_history[['id', 'teamId', 'roster_experience']], 
    left_on=['id', 'radiantTeamId'], 
    right_on=['id', 'teamId'], 
    how='left'
).rename(columns={'roster_experience': 'rad_roster_exp'}).drop('teamId', axis=1)

# Merge for Dire
df = df.merge(
    roster_history[['id', 'teamId', 'roster_experience']], 
    left_on=['id', 'direTeamId'], 
    right_on=['id', 'teamId'], 
    how='left'
).rename(columns={'roster_experience': 'dire_roster_exp'}).drop('teamId', axis=1)

In [31]:
## Calculating lane longevity
df_roster = df_roster.sort_values(['startDateTime', 'matchId'])
lanes = []
for (match_id, is_radiant), group in df_roster.groupby(['matchId', 'isRadiant']):
    pos_map = group.set_index('position')['steamAccountId'].to_dict()
    if 'POSITION_1' in pos_map and 'POSITION_5' in pos_map:
        lanes.append({
            'matchId': match_id,
            'pair': tuple(sorted([pos_map['POSITION_1'], pos_map['POSITION_5']])),
            'lane_type': 'Safelane',
            'timestamp': group['startDateTime'].iloc[0],
            'isRadiant': is_radiant
        })
        
    # Offlane Duo (Pos 3 & 4)
    if 'POSITION_3' in pos_map and 'POSITION_4' in pos_map:
        lanes.append({
            'matchId': match_id,
            'pair': tuple(sorted([pos_map['POSITION_3'], pos_map['POSITION_4']])),
            'lane_type': 'Offlane',
            'timestamp': group['startDateTime'].iloc[0],
            'isRadiant': is_radiant
        })

lane_df = pd.DataFrame(lanes)
lane_df['games_together_count'] = lane_df.groupby('pair').cumcount()
rad_safelane_long = lane_df[(lane_df['lane_type'] == 'Safelane') & (lane_df['isRadiant'] == True)].rename(
    {'games_together_count': 'radiant_safelane_long'}, axis=1
)
rad_offlane_long = lane_df[(lane_df['lane_type'] == 'Offlane') & (lane_df['isRadiant'] == True)].rename(
    {'games_together_count': 'radiant_offlane_long'}, axis=1
)
dire_safelane_long = lane_df[(lane_df['lane_type'] == 'Safelane') & (lane_df['isRadiant'] == False)].rename(
    {'games_together_count': 'dire_safelane_long'}, axis=1
)
dire_offlane_long = lane_df[(lane_df['lane_type'] == 'Offlane') & (lane_df['isRadiant'] == False)].rename(
    {'games_together_count': 'dire_offlane_long'}, axis=1
)

df = pd.concat(
    [
        df.reset_index(), 
        rad_safelane_long.iloc[:, -1].reset_index(), 
        rad_offlane_long.iloc[:, -1].reset_index(), 
        dire_safelane_long.iloc[:, -1].reset_index(), 
        dire_offlane_long.iloc[:, -1].reset_index()
    ], 
    axis=1
)